# Visualize Speech

Load a speech recording, build a spectrogram, extract MFCCs, and track pitch.

In [ ]:
# Install dependencies if needed (e.g. on Colab)
# %pip install pyvoicebox

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display

# Download a sample speech clip
url = "http://festvox.org/cmu_arctic/cmu_arctic/cmu_us_bdl_arctic/wav/arctic_a0001.wav"
wav_path = "arctic_a0001.wav"
if not os.path.exists(wav_path):
    print("Downloading sample speech...")
    urllib.request.urlretrieve(url, wav_path)

signal, fs = sf.read(wav_path)
duration = len(signal) / fs
print(f"Loaded: {wav_path} ({fs} Hz, {duration:.2f}s, {len(signal)} samples)")
print("Content: \"Author of the danger trail, Philip Steels, etc.\"")
display(Audio(signal, rate=fs))

## Waveform

In [ ]:
t = np.arange(len(signal)) / fs

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(t, signal, linewidth=0.3, color='#3f51b5')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Speech waveform')
ax.set_xlim(0, duration)
plt.tight_layout()
plt.show()

## Spectrogram

Built using pyvoicebox primitives: `v_enframe` chops the signal into overlapping frames, `v_windows` applies a Hamming window, and `v_rfft` computes the FFT.

The result shows frequency content over time. Bright horizontal bands are **formants** — resonances of the vocal tract. Vertical striations correspond to individual glottal pulses in voiced speech.

In [ ]:
from pyvoicebox import v_enframe, v_rfft, v_windows

frame_len = int(0.025 * fs)   # 25 ms
frame_hop = int(0.010 * fs)   # 10 ms

frames, frame_times, _ = v_enframe(signal, frame_len, frame_hop)
win = v_windows(3, frame_len).flatten()  # Hamming
spectra = np.array([v_rfft(f * win) for f in frames])
mag_db = 20 * np.log10(np.abs(spectra) + 1e-10)

time_axis = np.arange(mag_db.shape[0]) * frame_hop / fs
freq_axis = np.linspace(0, fs / 2, mag_db.shape[1])

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.pcolormesh(time_axis, freq_axis, mag_db.T, cmap='magma', shading='auto')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Spectrogram')
ax.set_ylim(0, fs / 2)
plt.colorbar(im, ax=ax, label='dB', pad=0.02)
plt.tight_layout()
plt.show()

## MFCCs

`v_melcepst` extracts Mel-frequency cepstral coefficients. The mode string `'M0'` means: **M**el scale, include the **0**th coefficient (log energy).

MFCCs are a compact representation of the spectral envelope. The lower coefficients capture broad spectral shape (vowel identity), while higher ones capture finer detail. They're the standard feature for speech and speaker recognition.

In [ ]:
from pyvoicebox import v_melcepst

mfcc, tc = v_melcepst(signal, fs, 'M0', 12)
print(f"MFCC shape: {mfcc.shape}  (frames x coefficients)")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(mfcc.T, aspect='auto', origin='lower', cmap='inferno',
               interpolation='nearest')
ax.set_xlabel('Frame')
ax.set_ylabel('MFCC coefficient')
ax.set_title('Mel-Frequency Cepstral Coefficients')
plt.colorbar(im, ax=ax, pad=0.02)
plt.tight_layout()
plt.show()

## Pitch tracking

`v_fxpefac` is the PEFAC pitch tracker — it returns the fundamental frequency (F0) per frame and a voicing probability.

In the plot below, green dots on the spectrogram show detected F0. The voicing probability plot shows where the algorithm detects voiced speech (vowels, nasals) vs unvoiced (fricatives, silence). Values near 1.0 indicate confident voicing.

In [ ]:
from pyvoicebox import v_fxpefac

pitch, pitch_times, voicing = v_fxpefac(signal, fs)

# Only plot voiced frames (F0 > 0)
voiced = pitch > 0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Spectrogram with pitch overlay
ax1.pcolormesh(time_axis, freq_axis, mag_db.T, cmap='magma', shading='auto')
ax1.scatter(pitch_times[voiced], pitch[voiced], c='#4caf50', s=8, zorder=5, label='F0')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title('Spectrogram with pitch contour')
ax1.set_ylim(0, 1000)
ax1.legend(loc='upper right')

# Voicing probability
ax2.fill_between(pitch_times, voicing, alpha=0.4, color='#e91e63')
ax2.plot(pitch_times, voicing, color='#e91e63', linewidth=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Voicing probability')
ax2.set_title('Voice activity')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()